# IPL Intelligence Engine — Full Pipeline
End-to-end walkthrough: load data → engineer features → train → evaluate → save models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

## 1. Load data

**Option A — from Kaggle (dataset: [chaitu20/ipl-dataset2008-2025](https://www.kaggle.com/datasets/chaitu20/ipl-dataset2008-2025/data))**

Ensure you have `kaggle.json` in `~/.kaggle/`, then run the cell below. It will download the dataset to `data/kaggle_ipl/` and load it with the correct schema.

**Option B — from a local CSV**

Use the second cell and set `DATA_PATH` to your file.

In [ ]:
# Option A: Load from Kaggle (downloads to data/kaggle_ipl/ if missing)
from data.kaggle_ipl import load_ipl_from_kaggle
df = load_ipl_from_kaggle(download_if_missing=True)
df.head(3)

In [ ]:
# Option B: Or load from a local CSV (run this instead of Option A if you have your own file)
# from data.loader import load_ipl
# df = load_ipl('../your_ipl_data.csv')
# df.head(3)

## 2. Toss alpha decay analysis

In [ ]:
from models.toss_alpha_decay import compute_toss_alpha_decay, print_decay_report, significant_venues

toss_alpha_df = compute_toss_alpha_decay(df)
print_decay_report(toss_alpha_df)

print('\nVenues with significant toss edge in modern era:')
print(significant_venues(toss_alpha_df, era='modern', min_edge=0.04))

## 3. Build pre-match features

In [ ]:
from models.pre_match_features import build_pre_match_features

pre_df = build_pre_match_features(df)
print(f'Pre-match features shape: {pre_df.shape}')
pre_df[['batting_team','bowling_team','elo_delta','form_delta','toss_edge_score','team_a_won']].head(5)

## 4. Build pressure index features (chase innings)

In [ ]:
from models.pressure_index import build_pressure_features

pressure_df = build_pressure_features(df)
print(f'Pressure features shape: {pressure_df.shape}')
print(f'Collapse rate: {pressure_df["collapse_next_30"].mean()*100:.1f}%')
pressure_df[['over','rrr','rrr_gap','wickets_fallen','pressure_index','collapse_next_30']].head(5)

## 5. Train the collapse predictor

In [ ]:
from models.pressure_index import train_collapse_model

collapse_model, collapse_cal, collapse_predict = train_collapse_model(pressure_df)

## 6. Train the unified match predictor

In [ ]:
from models.unified_predictor import train_all

lgb_pre, lgb_in, meta, pre_calibrator = train_all(pre_df, pressure_df)

## 7. Full evaluation

In [ ]:
from evaluation.evaluate import run_full_evaluation

test_pre = pre_df[pre_df['year'] >= 2023]
test_in  = pressure_df[pressure_df['season'] >= 2023]

results = run_full_evaluation(lgb_pre, lgb_in, meta, test_pre, test_in)

## 8. Save models

In [ ]:
import pickle, os
os.makedirs('../models/saved', exist_ok=True)

for name, obj in [('lgb_pre', lgb_pre), ('lgb_in', lgb_in),
                  ('meta', meta), ('toss_alpha', toss_alpha_df)]:
    with open(f'../models/saved/{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved {name}.pkl')

print('\nAll models saved. Launch the dashboard with:')
print('  python3 -m streamlit run dashboard/app.py')

## 9. Live prediction example

In [ ]:
from models.unified_predictor import predict_match
from models.pressure_index import live_collapse_probability

# Pre-match prediction
pre_feats = {
    'toss_edge_score': 0.03,
    'elo_delta': 80.0,
    'form_delta': 0.15,
    'h2h_venue_wr': 0.58,
    'is_playoff': 0,
    'match_number_norm': 0.4,
}
print('Pre-match prediction:')
print(predict_match(lgb_pre, lgb_in, meta, pre_feats))

# Live collapse risk (over 14, team needs 48 off 36 balls, 4 down)
print('\nCollapse risk at over 14:')
print(live_collapse_probability(
    runs_needed=48, balls_remaining=36, wickets_fallen=4,
    batter_sr_recent=95, partnership_balls=8, bowler_economy=7.2,
    model=collapse_model, calibrator=collapse_cal
))